# Restricted Boltzmann Machine: エネルギーで分布を表す

RBMは、可視変数と隠れ変数の組にエネルギーを割り当て、低エネルギーの状態を起こりやすくするモデルである。


## このノートの読み方

想定読者: 二値変数、sigmoid、確率分布、MLPの重みを理解した学生。

MLPの次に読む教材として、直感、数式、shape、コード、HTMLアニメーション、`Trainer`学習例を往復しながら読む。


## MLPからの橋渡し

MLPは入力から出力への関数を学ぶ。RBMは状態の起こりやすさをエネルギーで学び、データ状態とモデル状態を比べる。


## 到達目標

- エネルギー関数を説明できる
- `p(h|v)`と`p(v|h)`を計算できる
- CD-1の正相・負相を説明できる


## 重要語句

- `visible`: 観測される変数
- `hidden`: 特徴を表す隠れ変数
- `free energy`: hを周辺化したvのエネルギー


## 準備

すべてのコードは小さなテンソルで概念を確認するためのものです。長い学習は行いません。


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
from jaxtyping import Float
from torch import nn
from torch.utils.data import Dataset
import transformers
from transformers import Trainer, TrainingArguments

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


## shape表

数式を読む前に、どのテンソルがどのshapeを持つかを固定する。

| 記号 | shape | 意味 |
|---|---|---|
| v | (B, n_visible) | 可視状態 |
| h | (B, n_hidden) | 隠れ状態 |
| W | (n_visible, n_hidden) | 層間結合 |


## レビュー指摘を踏まえた補強

| 観点 | 補足 |
|---|---|
| エネルギーから確率へ | `p(v,h)=exp(-E(v,h))/Z`で低エネルギー状態ほど確率が高い。`Z`が全状態の和なので学習が難しい。 |
| 正相と負相 | 正相はデータから得た`vh^T`相関を強める。負相はモデルが作ったサンプルの相関を弱める。 |
| free energy | 隠れ変数を周辺化した`F(v)`を使うと、観測`v`の起こりやすさを比較しやすい。 |
| Trainer例の特殊性 | 教師ラベルを当てるのではなく、自由エネルギー差の近似をlossとして返す。通常の分類Trainerとは意味が違う。 |
| バイオ用途 | 二値化した変異パターンや発現有無の共起を、低エネルギー状態として覚える古典的な見方になる。 |


## Energy

低エネルギーほど起こりやすい。

$$
E(v,h)=-b^\top v-c^\top h-v^\top Wh
$$


## Conditional sampling

層内結合がないため、条件付きで各ユニットを独立にサンプルできる。

$$
p(h_j=1|v)=\sigma(c_j+W_{:j}^\top v)
$$


## Contrastive Divergence

データ側の相関を強め、モデル側の相関を弱める。

$$
\Delta W\propto\langle vh^\top\rangle_{\mathrm{data}}-\langle vh^\top\rangle_{\mathrm{model}}
$$


## 小さいテンソルで確認する

次のコードは、上の式がどのshapeを返すかを確認するための最小例である。


In [ ]:
v = torch.tensor([[1., 0., 1.]])
W = torch.randn(3, 2)
c = torch.zeros(2)
prob_h = torch.sigmoid(v @ W + c)
h = torch.bernoulli(prob_h)
prob_v = torch.sigmoid(h @ W.T)
print("p(h|v):", prob_h.round(decimals=3))
print("sample h:", h)
print("p(v|h):", prob_v.round(decimals=3))


## 難所HTMLスライド

数式だけでは混ざりやすい箇所を、スライド形式で確認する。各スライドでは入力shape、計算、lossまたは生成手順への接続を1つずつ見る。


<p><a href="../demos/restricted-boltzmann-machine_difficulty_slides.html?v=20260522" target="_blank" rel="noopener">別タブで難所スライドを開く</a>（リポジトリ内: <code>demos/restricted-boltzmann-machine_difficulty_slides.html</code>）</p>
<iframe
  src="../demos/restricted-boltzmann-machine_difficulty_slides.html?v=20260522"
  width="100%"
  height="720"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="Restricted Boltzmann Machine: エネルギーで分布を表す difficulty slides"
></iframe>


## HTMLアニメーションで確認する

以下のHTMLは`teaching-html-animation` skillの方針に合わせ、各状態を式・shape・コード上の概念に結びつけている。


### bipartite graph animation

- 学習目標: 層内結合なしの二部グラフ
- 誤解の防止: 普通の全結合NNと思う

対応する式:

$$
v^\top Wh
$$


<p><a href="../demos/restricted-boltzmann-machine_bipartite.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/restricted-boltzmann-machine_bipartite.html</code>）</p>
<iframe
  src="../demos/restricted-boltzmann-machine_bipartite.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="bipartite graph animation"
></iframe>


### Gibbs v to h to v animation

- 学習目標: v->h->v'の往復
- 誤解の防止: 一方向のforwardだと思う

対応する式:

$$
v\to h\to v'
$$


<p><a href="../demos/restricted-boltzmann-machine_gibbs.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/restricted-boltzmann-machine_gibbs.html</code>）</p>
<iframe
  src="../demos/restricted-boltzmann-machine_gibbs.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="Gibbs v to h to v animation"
></iframe>


### positive negative phase animation

- 学習目標: 正相と負相を比べる
- 誤解の防止: CDを通常のbackpropと思う

対応する式:

$$
\langle vh^\top\rangle_{data}-\langle vh^\top\rangle_{model}
$$


<p><a href="../demos/restricted-boltzmann-machine_cd_phase.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/restricted-boltzmann-machine_cd_phase.html</code>）</p>
<iframe
  src="../demos/restricted-boltzmann-machine_cd_phase.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="positive negative phase animation"
></iframe>


### energy landscape animation

- 学習目標: データの谷を低くする
- 誤解の防止: エネルギーの意味が分からない

対応する式:

$$
p(v,h)\propto\exp(-E(v,h))
$$


<p><a href="../demos/restricted-boltzmann-machine_energy_landscape.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/restricted-boltzmann-machine_energy_landscape.html</code>）</p>
<iframe
  src="../demos/restricted-boltzmann-machine_energy_landscape.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="energy landscape animation"
></iframe>


## `Trainer`で学習する

この章の`Trainer`例は、汎用MSE回帰ではなく、`Restricted Boltzmann Machine: エネルギーで分布を表す`固有のデータ形式とlossを返す。基本は標準の`Trainer(model, args, train_dataset)`を使い、`forward`が`loss`と`logits`を返す形にそろえる。GANはD/Gでoptimizerを分ける必要があるため`Trainer`を継承した交互更新デモ、DBMは平均場CDサロゲートとして扱う。


In [ ]:
class TinyBinaryDataset(Dataset):
    def __init__(self, n_samples: int = 32, n_visible: int = 6) -> None:
        base = torch.tensor([[1., 1., 1., 0., 0., 0.], [0., 0., 0., 1., 1., 1.]])
        ids = torch.randint(0, 2, (n_samples,))
        self.v = base[ids]

    def __len__(self) -> int:
        return len(self.v)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        return {"v": self.v[index]}


class TrainerRBM(nn.Module):
    def __init__(self, n_visible: int = 6, n_hidden: int = 3) -> None:
        super().__init__()
        self.W = nn.Parameter(torch.randn(n_visible, n_hidden) * 0.01)
        self.b = nn.Parameter(torch.zeros(n_visible))
        self.c = nn.Parameter(torch.zeros(n_hidden))

    def free_energy(self, v: torch.Tensor) -> torch.Tensor:
        hidden_term = torch.sum(torch.nn.functional.softplus(v @ self.W + self.c), dim=1)
        visible_term = v @ self.b
        return -visible_term - hidden_term

    def forward(self, v: torch.Tensor) -> dict[str, torch.Tensor]:
        with torch.no_grad():
            h_prob = torch.sigmoid(v @ self.W + self.c)
            h = torch.bernoulli(h_prob)
            v_prob = torch.sigmoid(h @ self.W.T + self.b)
            v_negative = torch.bernoulli(v_prob)
        loss = torch.mean(self.free_energy(v) - self.free_energy(v_negative))
        return {"loss": loss, "logits": v_prob, "reconstruction": v_prob}


training_args = TrainingArguments(
    output_dir="./results/restricted-boltzmann-machine_trainer_demo",
    max_steps=1,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_strategy="no",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    seed=SEED,
    use_cpu=not torch.cuda.is_available(),
)

trainer = Trainer(model=TrainerRBM(), args=training_args, train_dataset=TinyBinaryDataset())
train_output = trainer.train()
print("RBM CD-1 Trainer loss:", train_output.training_loss)


## 生成モデル間の比較

| モデル | 学習目的 | 尤度 | 生成手順 | 代表的な弱点 |
|---|---|---|---|---|
| VAE/IWAE | ELBO / IWAE bound | 下界 | Decoderにzを入れる | ぼやけ、推論分布の設計 |
| GAN | Dをだます | 通常は不可 | G(z)を一発生成 | mode collapse、不安定 |
| Flow | NLL | 厳密 | 可逆変換の順方向 | 可逆層の制約 |
| RBM/DBM | エネルギー差 | 分配関数が困難 | Gibbs sampling | 近似推論が重い |
| Diffusion/DDPM | score/noise予測 | 目的により異なる | 多段denoising | samplingが遅い |


## 発展課題

- CD-kを変える
- persistent CDを調べる
- 現代EBMとscore matchingへ接続する


## 確認問題

- RBMのrestrictedとは何を制限しているか。
- CD-1の正相と負相の違いを書く。


## まとめ

- MLPから何が変わったのかを、shapeとlossで確認する。
- HTMLアニメーションは式の代わりではなく、式とコードを読むための補助である。
- `Trainer`は学習ループを隠すが、`Dataset`のキー、`forward`の引数、`loss`の意味は必ず確認する。
